# QuantumCircuit.jl Phase 3A Closed-System Dynamics Walkthrough

**Audience**
- Julia users who already know the static and sweep workflows and want to start time-domain simulations.

**Prerequisites**
- Basic Julia syntax.
- Familiarity with `CompositeSystem`, `build_model`, and the Phase 2 analysis helpers.

**What you will learn**
- How to build a named product state with `basis_state`.
- How to request expectation values with `ObservableSpec`.
- How to run closed-system evolution with `evolve`.
- How to add a narrow subsystem-local drive with `SubsystemDrive`.


## Outline

1. Activate the local package environment.
2. Build a basis state and run free evolution in a simple system.
3. Read expectation-value traces from `DynamicsResult`.
4. Add a subsystem-local drive and inspect the response.
5. Review the current limits of the Phase 3A dynamics layer.
6. Try a short exercise.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


## Step 1 - Build a named product state

The new `basis_state` helper avoids manual tensor-product bookkeeping. Unspecified subsystems default to level `0`.


In [3]:
resonator = Resonator(:r1; ω = 6.0, dim = 2)
sys = CompositeSystem(resonator)
model = build_model(sys)
ψ0 = basis_state(sys; r1 = 1)

(
    subsystem_order = model.subsystem_order,
    dimensions = model.dimensions,
    initial_state_length = length(ψ0.data),
    initial_state = ψ0.data,
)


(subsystem_order = [:r1], dimensions = Dict(:r1 => 2), initial_state_length = 2, initial_state = ComplexF64[0.0 + 0.0im, 1.0 + 0.0im])

## Step 2 - Run free evolution with a named observable

`ObservableSpec` describes which embedded operator to measure. Here we track the resonator number operator, which should stay constant under free evolution from a number eigenstate.


In [4]:
tlist = collect(range(0.0, 1.0; length = 11))
result = evolve(
    sys,
    ψ0,
    tlist;
    observables = [ObservableSpec(:nr, :r1, :n)],
)

nr_trace = observable_trace(result, :nr)

(
    saved_times = result.times,
    number_trace = nr_trace.values,
    final_state = final_state(result).data,
)


(saved_times = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], number_trace = ComplexF64[1.0 + 0.0im, 1.0000000000565694 + 0.0im, 1.0000000003335376 + 0.0im, 1.0000000007596401 + 0.0im, 1.0000000018769613 + 0.0im, 1.0000000028940423 + 0.0im, 1.0000000049204216 + 0.0im, 1.0000000070006083 + 0.0im, 1.0000000089220489 + 0.0im, 1.0000000116461476 + 0.0im, 1.0000000136695126 + 0.0im], final_state = ComplexF64[0.0 + 0.0im, 0.9601702927555886 + 0.2794155016801319im])

## Step 3 - Read the `DynamicsResult`

`DynamicsResult` keeps the saved states, normalized observable traces, and the underlying solver result for debugging.


In [5]:
(
    state_count = length(result.states),
    observable_labels = [trace.label for trace in result.observables],
    raw_solver_type = typeof(result.solver_result),
)


(state_count = 11, observable_labels = [:nr], raw_solver_type = QuantumToolbox.TimeEvolutionSol{Vector{Float64}, Vector{Float64}, Vector{QuantumToolbox.QuantumObject{QuantumToolbox.Ket, QuantumToolbox.ProductDimensions{1, 1, Tuple{QuantumToolbox.HilbertSpace}, Tuple{QuantumToolbox.HilbertSpace}}, Vector{ComplexF64}}}, Matrix{ComplexF64}, SciMLBase.ReturnCode.T, OrdinaryDiffEqVerner.Vern7{typeof(OrdinaryDiffEqCore.trivial_limiter!), typeof(OrdinaryDiffEqCore.trivial_limiter!), Static.False}, Float64})

## Step 4 - Add a subsystem-local drive

Phase 3A intentionally keeps drives narrow. A `SubsystemDrive` adds one time-dependent operator term on a named subsystem.


In [6]:
drive = SubsystemDrive(
    :r1_x_drive,
    :r1,
    :x,
    (p, t) -> p.Ω * cos(p.ωd * t),
)

driven_result = evolve(
    sys,
    basis_state(sys; r1 = 0),
    collect(range(0.0, 6.0; length = 61));
    drives = [drive],
    observables = [ObservableSpec(:nr, :r1, :n)],
    params = (; Ω = 0.35, ωd = 1.0),
)

driven_trace = observable_trace(driven_result, :nr)

(
    first_values = driven_trace.values[1:10],
    max_population = maximum(real.(driven_trace.values)),
    final_population = real(driven_trace.values[end]),
)


(first_values = ComplexF64[0.0 + 0.0im, 0.0011842581924863926 + 0.0im, 0.004275616799462683 + 0.0im, 0.008081213408778735 + 0.0im, 0.011166631619526694 + 0.0im, 0.012430195035442433 + 0.0im, 0.011515474898566422 + 0.0im, 0.008894549246686916 + 0.0im, 0.005605605241422755 + 0.0im, 0.0027808860558008073 + 0.0im], max_population = 0.014057030597293558, final_population = 0.006557223882065279)

## Step 5 - Compare free and driven dynamics

The free-evolution trace stays flat because the initial state is already an eigenstate of the static Hamiltonian. The driven trace changes because the extra `x` term mixes the two levels.


In [7]:
collect(zip(result.times, real.(nr_trace.values), real.(driven_trace.values[1:length(result.times)])))


11-element Vector{Tuple{Float64, Float64, Float64}}:
 (0.0, 1.0, 0.0)
 (0.1, 1.0000000000565694, 0.0011842581924863926)
 (0.2, 1.0000000003335376, 0.004275616799462683)
 (0.3, 1.0000000007596401, 0.008081213408778735)
 (0.4, 1.0000000018769613, 0.011166631619526694)
 (0.5, 1.0000000028940423, 0.012430195035442433)
 (0.6, 1.0000000049204216, 0.011515474898566422)
 (0.7, 1.0000000070006083, 0.008894549246686916)
 (0.8, 1.0000000089220489, 0.005605605241422755)
 (0.9, 1.0000000116461476, 0.0027808860558008073)
 (1.0, 1.0000000136695126, 0.0011852497989352068)

## Pitfalls and Current Limits

**Current scope**
- `evolve` currently wraps closed-system `sesolve` workflows.
- `ObservableSpec` is limited to the built-in local operators `:a`, `:adag`, `:n`, `:x`, and `:y`.
- Drives are intentionally subsystem-local for the first Phase 3A slice.

**Not yet in scope**
- Lindblad dynamics.
- collapse operators.
- steady-state solvers.


## Exercise

Change the drive amplitude and frequency and inspect how the number trace changes.

Suggested experiments:
1. Increase `Ω` and check whether the maximum population rises faster.
2. Detune `ωd` away from `1.0` and compare the final population.
3. Replace the number trace with an `:x` trace and compare the signal shape.


In [8]:
exercise_drive = SubsystemDrive(
    :r1_x_drive,
    :r1,
    :x,
    (p, t) -> p.Ω * cos(p.ωd * t),
)

exercise_result = evolve(
    sys,
    basis_state(sys; r1 = 0),
    collect(range(0.0, 6.0; length = 61));
    drives = [exercise_drive],
    observables = [ObservableSpec(:xr, :r1, :x)],
    params = (; Ω = 0.50, ωd = 0.85),
)

exercise_trace = observable_trace(exercise_result, :xr)
(
    times = exercise_trace.times[1:10],
    x_values = exercise_trace.values[1:10],
)


(times = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], x_values = ComplexF64[0.0 + 0.0im, -0.029068549171443924 + 0.0im, -0.10563942760311579 + 0.0im, -0.20163901693039654 + 0.0im, -0.2816569113467715 + 0.0im, -0.3156603277538426 + 0.0im, -0.28973405109418887 + 0.0im, -0.21100176324745812 + 0.0im, -0.10500920611815989 + 0.0im, -0.006567077892166642 + 0.0im])